In [3]:
import os
import glob
from typing import List, Dict, Any, Set

import numpy as np
import pandas as pd
import redis
import langchain

from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS

#from langchain.cache import RedisCache  # works in many installs; if missing, we fallback below

import langchain
from langchain_redis import RedisCache
# ----------------------------
# Load .env
# ----------------------------
load_dotenv()


# ----------------------------
# Config
# ----------------------------
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

REDIS_URL = os.getenv("REDIS_URL", "redis://localhost:6379")
REDIS_CACHE_TTL_SECONDS = int(os.getenv("REDIS_CACHE_TTL_SECONDS", "3600"))

DATA_DIR = os.getenv("DATA_DIR", "./pdf_data")

EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-large")
CHAT_MODEL = os.getenv("CHAT_MODEL", "gpt-4o-mini")

CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "900"))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "150"))
TOP_K = int(os.getenv("TOP_K", "4"))

EVAL_CSV_PATH = os.getenv("EVAL_CSV_PATH", "./eval_dataset.csv")
EVAL_OUT_CSV_PATH = os.getenv("EVAL_OUT_CSV_PATH", "./eval_results.csv")
EVAL_QUESTION_COL = os.getenv("EVAL_QUESTION_COL", "question")
EVAL_REFERENCE_COL = os.getenv("EVAL_REFERENCE_COL", "reference_answer")
EVAL_MAX_ROWS = int(os.getenv("EVAL_MAX_ROWS", "0"))  # 0 => all rows


# ----------------------------
# Env checks + Redis checks
# ----------------------------
def ensure_env():
    if not OPENAI_API_KEY:
        raise RuntimeError("Missing OPENAI_API_KEY (set it in .env).")


def assert_redis_up(redis_url: str):
    r = redis.from_url(redis_url)
    try:
        ok = r.ping()
        if ok is not True:
            raise RuntimeError("Redis PING failed.")
    except Exception as e:
        raise RuntimeError(f"Cannot connect to Redis at {redis_url}. Error: {repr(e)}")


def enable_langchain_cache(redis_url: str, ttl_seconds: int):
    """
    LangChain cache API varies across versions.
    For LangChain 1.0.3, setting langchain.llm_cache is reliable.
    TTL support depends on cache implementation; if ttl isn't supported, it will be ignored.
    """
    try:
        # If available, use LangChain's RedisCache
        # Some builds don't accept ttl; handle both.
        try:
            langchain.llm_cache = RedisCache(redis_url=redis_url, ttl=ttl_seconds)
        except TypeError:
            langchain.llm_cache = RedisCache(redis_url=redis_url)
    except Exception:
        # If the import/path is weird in your environment, you can skip caching gracefully
        # but we try hard not to.
        raise RuntimeError(
            "Failed to enable Redis cache. Ensure 'redis' and 'langchain' are installed "
            "and that langchain.cache.RedisCache is available."
        )


# ----------------------------
# PDF loading + chunking
# ----------------------------
def find_pdfs(data_dir: str) -> List[str]:
    pattern = os.path.join(data_dir, "**", "*.pdf")
    return sorted(glob.glob(pattern, recursive=True))


def load_pdf_docs(pdf_paths: List[str], base_dir: str) -> List[Document]:
    """
    Loads PDFs into Documents (one Document per page) and normalizes metadata.
    """
    docs: List[Document] = []
    for path in pdf_paths:
        loader = PyPDFLoader(path)
        loaded = loader.load()
        rel = os.path.relpath(path, base_dir)

        for d in loaded:
            d.metadata = d.metadata or {}
            d.metadata["source"] = rel
            # PyPDFLoader typically sets page in metadata as 'page'
            docs.append(d)

    return docs


def chunk_docs(docs: List[Document]) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
    return splitter.split_documents(docs)


def format_docs_with_sources(docs: List[Document]) -> str:
    out = []
    for i, d in enumerate(docs, start=1):
        src = d.metadata.get("source", "unknown")
        page = d.metadata.get("page", None)
        page_str = f", page {page}" if page is not None else ""
        out.append(f"---\nChunk {i} [source: {src}{page_str}]\n{d.page_content}")
    return "\n".join(out)


def extract_sources(docs: List[Document]) -> List[str]:
    s: Set[str] = set()
    for d in docs:
        src = d.metadata.get("source", "unknown")
        page = d.metadata.get("page", None)
        if page is not None:
            s.add(f"{src} (page {page})")
        else:
            s.add(src)
    return sorted(s)


# ----------------------------
# Build FAISS Vector Store
# ----------------------------
def build_faiss_vector_store() -> FAISS:
    """
    Builds a FAISS vector store from all PDFs in DATA_DIR.
    """
    pdfs = find_pdfs(DATA_DIR)
    if not pdfs:
        raise RuntimeError(f"No PDFs found in {DATA_DIR}. Add PDFs and re-run.")

    raw_docs = load_pdf_docs(pdfs, base_dir=DATA_DIR)
    chunks = chunk_docs(raw_docs)

    embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
    vs = FAISS.from_documents(chunks, embeddings)

    print(f"Ingested {len(chunks)} chunks from {len(pdfs)} PDFs into FAISS.")
    return vs


# ----------------------------
# RAG Chain
# ----------------------------
def build_rag_chain(vector_store: FAISS):
    retriever = vector_store.as_retriever(search_kwargs={"k": TOP_K})

    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "You are a helpful assistant. Answer ONLY using the provided context. "
                "If the answer is not in the context, say you don't know. "
                "At the end include a short 'Sources:' list."
            ),
            ("human", "Question:\n{question}\n\nContext:\n{context}"),
        ]
    )

    llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)

    rag_chain = (
        {
            "question": RunnablePassthrough(),
            "context": retriever | RunnableLambda(format_docs_with_sources),
        }
        | prompt
        | llm
    )

    def get_sources(q: str) -> Dict[str, Any]:
        #docs = retriever.get_relevant_documents(q)
        docs = retriever.invoke(q) 
        return {"sources": extract_sources(docs)}

    sources_chain = RunnableLambda(get_sources)

    return rag_chain, sources_chain


# ----------------------------
# Evaluation helpers
# ----------------------------
def safe_text(x: Any) -> str:
    if x is None:
        return ""
    if isinstance(x, float) and np.isnan(x):
        return ""
    return str(x).strip()


def cosine_sim(vec_a: List[float], vec_b: List[float]) -> float:
    a = np.array(vec_a, dtype=np.float32)
    b = np.array(vec_b, dtype=np.float32)
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


# ----------------------------
# Main: CSV eval mode
# ----------------------------
def main():
    ensure_env()
    assert_redis_up(REDIS_URL)
    enable_langchain_cache(REDIS_URL, REDIS_CACHE_TTL_SECONDS)

    # Build FAISS index from PDFs
    vector_store = build_faiss_vector_store()

    # Build RAG
    rag_chain, sources_chain = build_rag_chain(vector_store)

    # Load evaluation CSV
    if not os.path.exists(EVAL_CSV_PATH):
        raise FileNotFoundError(f"CSV not found: {EVAL_CSV_PATH}")

    df = pd.read_csv(EVAL_CSV_PATH)

    if EVAL_QUESTION_COL not in df.columns or EVAL_REFERENCE_COL not in df.columns:
        raise ValueError(
            f"CSV must contain columns '{EVAL_QUESTION_COL}' and '{EVAL_REFERENCE_COL}'. "
            f"Found: {list(df.columns)}"
        )

    if EVAL_MAX_ROWS and EVAL_MAX_ROWS > 0:
        df = df.head(EVAL_MAX_ROWS).copy()

    # Embeddings for answer similarity
    emb = OpenAIEmbeddings(model=EMBEDDING_MODEL)

    results = []
    sims = []

    print(f"Evaluating {len(df)} rows from {EVAL_CSV_PATH} ...")

    for i, row in df.iterrows():
        question = safe_text(row[EVAL_QUESTION_COL])
        ref_answer = safe_text(row[EVAL_REFERENCE_COL])

        if not question:
            results.append(
                {
                    "row": int(i),
                    "question": question,
                    "reference_answer": ref_answer,
                    "rag_answer": "",
                    "cosine_similarity": np.nan,
                    "sources": "",
                    "error": "empty_question",
                }
            )
            continue

        try:
            answer_msg = rag_chain.invoke(question)
            rag_answer = safe_text(getattr(answer_msg, "content", str(answer_msg)))

            srcs = sources_chain.invoke(question).get("sources", [])
            srcs_str = "; ".join(srcs) if isinstance(srcs, list) else safe_text(srcs)

            # Cosine similarity between reference answer and RAG answer
            if ref_answer and rag_answer:
                ref_vec = emb.embed_query(ref_answer)
                rag_vec = emb.embed_query(rag_answer)
                sim = cosine_sim(ref_vec, rag_vec)
            else:
                sim = np.nan

            if not np.isnan(sim):
                sims.append(sim)

            results.append(
                {
                    "row": int(i),
                    "question": question,
                    "reference_answer": ref_answer,
                    "rag_answer": rag_answer,
                    "cosine_similarity": sim,
                    "sources": srcs_str,
                    "error": "",
                }
            )

            if (len(results) % 10) == 0:
                avg_so_far = float(np.mean(sims)) if sims else float("nan")
                print(f"Processed {len(results)}/{len(df)} | avg cosine so far: {avg_so_far:.4f}")

        except Exception as e:
            results.append(
                {
                    "row": int(i),
                    "question": question,
                    "reference_answer": ref_answer,
                    "rag_answer": "",
                    "cosine_similarity": np.nan,
                    "sources": "",
                    "error": repr(e),
                }
            )

    out_df = pd.DataFrame(results)
    out_df.to_csv(EVAL_OUT_CSV_PATH, index=False)

    valid_count = int(np.sum(~out_df["cosine_similarity"].isna()))
    mean_sim = float(out_df["cosine_similarity"].mean()) if valid_count else float("nan")
    median_sim = float(out_df["cosine_similarity"].median()) if valid_count else float("nan")

    print("\n=== Evaluation Summary ===")
    print(f"Rows processed: {len(out_df)}")
    print(f"Rows with valid similarity: {valid_count}")
    print(f"Mean cosine similarity: {mean_sim:.4f}" if valid_count else "Mean cosine similarity: n/a")
    print(f"Median cosine similarity: {median_sim:.4f}" if valid_count else "Median cosine similarity: n/a")
    print(f"Saved results to: {EVAL_OUT_CSV_PATH}")


#if __name__ == "__main__":
#    main()
main()

Ingested 70 chunks from 1 PDFs into FAISS.
Evaluating 50 rows from ./RA_FSM_QA.csv ...
Processed 10/50 | avg cosine so far: 0.5241
Processed 20/50 | avg cosine so far: 0.5104
Processed 30/50 | avg cosine so far: 0.5124
Processed 40/50 | avg cosine so far: 0.4861
Processed 50/50 | avg cosine so far: 0.4839

=== Evaluation Summary ===
Rows processed: 50
Rows with valid similarity: 50
Mean cosine similarity: 0.4839
Median cosine similarity: 0.4941
Saved results to: ./eval_results.csv
